In [ ]:
Objetivo: Crear una sesión llamada MiPrimerAnalisis con 1g de memoria para el executor y 8 particiones 
para shuffle, y mostrar su información.

In [1]:
from pyspark.sql import SparkSession

# 1. Detenemos cualquier sesión anterior para que aplique nuestra nueva configuración
try:
    spark.stop()
except:
    pass

# 2. Creamos la SparkSession personalizada
spark = (SparkSession.builder
    .appName("MiPrimerAnalisis")
    .config("spark.executor.memory", "1g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate())

# 3. Verificamos que los parámetros se aplicaron correctamente
print("Nombre de la App:", spark.sparkContext.appName)
print("Memoria del Executor:", spark.conf.get("spark.executor.memory"))
print("Particiones Shuffle:", spark.conf.get("spark.sql.shuffle.partitions"))

# 4. Mostramos la tarjeta de información de la sesión
spark

26/11/20 00:03:45 WARN Utils: Your hostname, valentin-ThinkPad-T440p resolves to a loopback address: 127.0.1.1; using 192.168.1.105 instead (on interface wlp3s0)
26/11/20 00:03:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/20 00:03:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/20 00:03:47 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Nombre de la App: MiPrimerAnalisis
Memoria del Executor: 1g
Particiones Shuffle: 8


In [2]:
sc = spark.sparkContext

# 1. Crear RDD a partir de la lista de números del 1 al 20
numeros = list(range(1, 21))
rdd_numeros = sc.parallelize(numeros)

# 2. Operaciones básicas (Acciones)
total_elementos = rdd_numeros.count()
primeros_5 = rdd_numeros.take(5)
suma_total = rdd_numeros.sum()
num_max = rdd_numeros.max()
num_min = rdd_numeros.min()

# 3. Mostrar resultados
print(f"Total de elementos: {total_elementos}")
print(f"Primeros 5: {primeros_5}")
print(f"Suma: {suma_total}")
print(f"Máximo: {num_max}, Mínimo: {num_min}")

Total de elementos: 20
Primeros 5: [1, 2, 3, 4, 5]
Suma: 210
Máximo: 20, Mínimo: 1


In [3]:
# Ejercicio 3: Transformaciones map y filter
numeros = sc.parallelize(range(1, 51))  # Números del 1 al 50

# 1. Multiplicar cada número por 3 (Transformación map)
multiplicados = numeros.map(lambda x: x * 3)

# 2. Filtrar solo los mayores a 50 (Transformación filter)
mayores_50 = multiplicados.filter(lambda x: x > 50)

# 3. Acciones para obtener resultados
total_mayores = mayores_50.count()
primeros_10 = mayores_50.take(10)

print(f"Total de números mayores a 50: {total_mayores}")
print(f"Primeros 10: {primeros_10}")

Total de números mayores a 50: 34
Primeros 10: [51, 54, 57, 60, 63, 66, 69, 72, 75, 78]


In [4]:
# Ejercicio 4: Pares Clave-Valor (reduceByKey)
ventas = [
    ("Laptop", 1200),
    ("Mouse", 25),
    ("Laptop", 1100),
    ("Teclado", 75),
    ("Mouse", 30),
    ("Laptop", 1300)
]

# 1. Crear RDD de pares (clave, valor)
rdd_ventas = sc.parallelize(ventas)

# 2. Total de ventas por producto usando reduceByKey
total_por_producto = rdd_ventas.reduceByKey(lambda a, b: a + b).collect()

# 3. Contar cuántas ventas hay de cada producto
conteo_ventas = rdd_ventas.map(lambda x: (x[0], 1)).reduceByKey(lambda a, b: a + b).collect()

# 4. Encontrar el producto con mayor total de ventas
mayor_venta = max(total_por_producto, key=lambda x: x[1])

print("Total por producto:", dict(total_por_producto))
print("Número de ventas:", dict(conteo_ventas))
print(f"Producto con mayor total: {mayor_venta[0]} ({mayor_venta[1]})")

Total por producto: {'Teclado': 75, 'Laptop': 3600, 'Mouse': 55}
Número de ventas: {'Teclado': 1, 'Laptop': 3, 'Mouse': 2}
Producto con mayor total: Laptop (3600)


In [ ]:
Celda para el Ejercicio 5: Lectura y Procesamiento de Archivo CSV (persons.csv)

In [5]:
import os

# 1. Crear carpeta DATOS y descargar persons.csv
!mkdir -p DATOS
!wget -q "https://github.com/josepgarcia/datos/raw/refs/heads/main/persons.csv" -O ./DATOS/persons.csv

# 2. Cargar usando sc.textFile() con ruta absoluta y prefijo file://
path_persons = "file://" + os.path.abspath("./DATOS/persons.csv")
rdd_persons = sc.textFile(path_persons)

# 3. Eliminar encabezado
header = rdd_persons.first()
rdd_sin_header = rdd_persons.filter(lambda linea: linea != header)

# 4. Extraer (nombre, edad) y filtrar mayores de 30
mayores_30 = (rdd_sin_header
    .map(lambda linea: linea.split(","))
    .map(lambda cols: (cols[0].strip(), int(cols[1].strip())))
    .filter(lambda persona: persona[1] > 30))

# 5. Mostrar resultados
lista_mayores = mayores_30.collect()
print("Personas mayores de 30:", ", ".join([f"{n} ({e})" for n, e in lista_mayores]))
print(f"Total: {len(lista_mayores)} personas")

Personas mayores de 30: David Johnson (35), Michael Wilson (40), Sarah Brown (32), James Martinez (38), William Anderson (33), Karen Hernandez (31), Christopher Taylor (37), Matthew Davis (36), Daniel Miller (34), Linda Martin (39)
Total: 10 personas


In [ ]:
Celda para el Ejercicio 6: Optimización - Evitar Múltiples split()

In [6]:
# Ejercicio 6: Optimización - Hacer split() una sola vez por línea
# En lugar de llamar 3 veces a rdd.map(lambda x: x.split(",")[i]),
# extraemos todos los campos en una sola pasada:

def parsear_persona(linea):
    columnas = linea.split(",")
    nombre = columnas[0].strip()
    edad = columnas[1].strip()
    ciudad = columnas[2].strip()
    return (nombre, edad, ciudad)

datos_optimizados = rdd_sin_header.map(parsear_persona)
print("Registros extraídos en una sola pasada:", datos_optimizados.collect())

Registros extraídos en una sola pasada: [('John Doe', '30', 'Male'), ('Jane Smith', '25', 'Female'), ('David Johnson', '35', 'Male'), ('Emily Davis', '28', 'Female'), ('Michael Wilson', '40', 'Male'), ('Sarah Brown', '32', 'Female'), ('Robert Lee', '29', 'Male'), ('Lisa Garcia', '27', 'Female'), ('James Martinez', '38', 'Male'), ('Jennifer Rodriguez', '26', 'Female'), ('William Anderson', '33', 'Male'), ('Karen Hernandez', '31', 'Female'), ('Christopher Taylor', '37', 'Male'), ('Mary Gonzalez', '24', 'Female'), ('Matthew Davis', '36', 'Male'), ('Patricia White', '29', 'Female'), ('Daniel Miller', '34', 'Male'), ('Elizabeth Jackson', '30', 'Female'), ('Joseph Harris', '28', 'Male'), ('Linda Martin', '39', 'Female')]


In [ ]:
Celda para el Ejercicio 7: Debugging - Código Corregido

In [7]:
# Ejercicio 7: Debugging - Corrección de los 4 errores
import os
from pyspark.sql import SparkSession

# Corrección 1: Crear SparkSession y obtener SparkContext (sc)
spark = SparkSession.builder.appName("Ejercicio7Corregido").getOrCreate()
sc = spark.sparkContext

# Corrección 2: Usar ruta absoluta con prefijo file://
path_csv = "file://" + os.path.abspath("./DATOS/persons.csv")
rdd = sc.textFile(path_csv)

# Corrección 3: Evitar collect() en archivos grandes; usar take() para muestra segura
muestra = rdd.take(5)
for linea in muestra:
    print(linea)

# Corrección 4: Cerrar la sesión al terminar para liberar recursos y el puerto 4040
spark.stop()
print("Sesión cerrada correctamente.")

26/11/20 05:45:49 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


name,age,gender,salary
John Doe,30,Male,50000
Jane Smith,25,Female,45000
David Johnson,35,Male,60000
Emily Davis,28,Female,52000
Sesión cerrada correctamente.
